In [4]:
from multiprocessing import set_start_method
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# **Must** happen before torch or vllm ever touches CUDA
set_start_method("spawn", force=True)
from torch.utils.data import DataLoader, TensorDataset
from datasets import interleave_datasets 
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerFast, PreTrainedModel
from transformers import Trainer, TrainingArguments
from trl import SFTConfig, SFTTrainer
from trl import setup_chat_format
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import load_dataset, Dataset
from concurrent.futures import ThreadPoolExecutor
from trl import DataCollatorForCompletionOnlyLM
import torch
from vllm import LLM, SamplingParams
from vllm.model_executor import set_random_seed as vllm_set_random_seed
from drgrpo_grader import r1_zero_reward_fn
import gc
from unittest.mock import patch
import wandb
import safetensors
import os
import json
import numpy as np
import time
import random 
import operator
import itertools
from transformers import TrainerCallback, TrainerState, TrainerControl
from copy import deepcopy


In [5]:
def tokenize_prompt_and_output(prompt_strs, output_strs, tokenizer):
    """prompt and output strings, and construct a mask that is 1 for the response tokens and 0 for other tokens (prompt or padding).
    Args:
    prompt_strs: list[str] List of prompt strings.
    output_strs: list[str] List of output strings.
    tokenizer: PreTrainedTokenizer Tokenizer to use for tokenization
    """
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    input_ids, labels, response_masks = [], [], []
    max_len = 0
    for i, (prompt, output) in enumerate(zip(prompt_strs, output_strs)):
        prompt_enc = tokenizer(prompt, add_special_tokens=False) #return_tensors="pt"
        output_enc = tokenizer(output, add_special_tokens=False)

        prompt_ids = prompt_enc["input_ids"]
        output_ids = output_enc["input_ids"]

        # Concatenate
        concat_ids = prompt_ids + output_ids
        max_len = max(max_len, len(concat_ids)) 
        #attention_mask = [1] * len(input_ids)
        response_mask = [0] * len(prompt_ids) + [1] * len(output_ids)
        input_ids.append(concat_ids[:-1])
        labels.append(concat_ids[1:])
        #attention_masks.append(attention_mask)
        response_masks.append(response_mask[1:])
    if max_len == 0:
        raise ValueError
    max_len -= 1

    for i in range(len(input_ids)):
        curr_len = len(input_ids[i])
        padding_len = max_len - curr_len
        input_ids[i].extend([tokenizer.pad_token_id]*padding_len)
        labels[i].extend([tokenizer.pad_token_id]*padding_len)
        response_masks[i].extend([0]*padding_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "response_mask": torch.tensor(response_masks, dtype=torch.float)
    }

def compute_entropy(logits: torch.Tensor):
    """Get the entropy of the next-token predictions (i.e., entropy over the vocabulary dimension).
    Args:
    logits: torch.Tensor Tensor of shape (batch_size, sequence_length, vocab_size)
    containing unnormalized logits.
    Returns:
    torch.Tensor Shape (batch_size, sequence_length). The entropy for each next-token
    prediction.
    """
    # sum(p log p ) -> p= e^xi / sum(e^x) -> logp = xi - log(sum(e^x))
    logz = torch.logsumexp(logits, dim=-1, keepdim=True)
    logp = logits - logz
    p = torch.exp(logp)
    entropy = - torch.sum(p * logp, dim=-1)
    return entropy



def get_response_log_probs(
    model: PreTrainedModel,
    input_ids: torch.Tensor,
    labels: torch.Tensor,
    return_token_entropy: bool = False,
    ) -> dict[str, torch.Tensor]:
    """ can be used in NLL loss.
    Args:
        model: PreTrainedModel HuggingFace model used for scoring (placed on the correct device
        and in inference mode if gradients should not be computed).
        input_ids: torch.Tensor shape (batch_size, sequence_length), concatenated prompt +
        response tokens as produced by your tokenization method.
        labels: torch.Tensor shape (batch_size, sequence_length), labels as produced by your
        tokenization method.
        return_token_entropy: bool If True, also return per-token entropy by calling
        compute_entropy.
    Returns:
        dict[str, torch.Tensor].
            "log_probs" shape (batch_size, sequence_length), conditional log-probabilities log pθ(xt | x<t).
            "token_entropy" optional, shape (batch_size, sequence_length), per-token entropy for each position (present only if return_token_entropy=True)."""
        

    #with torh.no_grad():
    output = model(input_ids)
    logits = output.logits ##batch_size, seq_len vocab
    log_prob = F.log_softmax(logits, dim=-1)

    # Get log-prob of the true next token at each position
    # Use torch.gather to index the log_probs at label positions
    # labels: (batch_size, seq_len)
    # log_probs: (batch_size, seq_len, vocab_size)
    log_probs_at_labels = torch.gather(log_prob, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1) #batch_size, seq_len

    res = dict()
    res['log_probs'] = log_probs_at_labels
    if return_token_entropy:
        with torch.no_grad():
            res['token_entropy'] = compute_entropy(logits)
        #.detach().cpu()
    return res

In [5]:
def compute_group_normalized_rewards(
    reward_fn,
    rollout_responses,
    repeated_ground_truths,
    group_size,
    advantage_eps,
    normalize_by_std,
    ):
    """
    Compute rewards for each group of rollout responses, normalized by the group size.
    Args:
    reward_fn: Callable[[str, str], dict[str, float]] Scores the rollout responses against
    the ground truths, producing a dict with keys "reward", "format_reward", and
    "answer_reward".
    rollout_responses: list[str] Rollouts from the policy. The length of this list is
    rollout_batch_size = n_prompts_per_rollout_batch * group_size.
    repeated_ground_truths: list[str] The ground truths for the examples. The length of this
    list is rollout_batch_size, because the ground truth for each example is repeated
    group_size times.
    group_size: int Number of responses per question (group).
    advantage_eps: float Small constant to avoid division by zero in normalization.
    normalize_by_std: bool If True, divide by the per-group standard deviation; otherwise
    subtract only the group mean.
    Returns:
    tuple[torch.Tensor, torch.Tensor, dict[str, float]].
    advantages shape (rollout_batch_size,). Group-normalized rewards for each rollout
    response.
    raw_rewards shape (rollout_batch_size,). Unnormalized rewards for each rollout
    response.
    metadata your choice of other statistics to log (e.g. mean, std, max/min of rewards).
    """
    rollout_batch_size = len(rollout_responses)
    n_prompts_per_rollout_batch = rollout_batch_size // group_size
    # ground_truths = [[gt]*group_size for gt in repeated_ground_truths]
    # ground_truths = list(itertools.chain.from_iterable(ground_truths))
    rewards = []
   
    for response, gt in zip(rollout_responses, repeated_ground_truths):
        reward = reward_fn(response, gt)
        rewards.append(reward['reward'])
    raw_rewards = torch.Tensor(rewards)
   
    rewards = raw_rewards.view(n_prompts_per_rollout_batch, group_size)
    mean_rewards =  torch.mean(rewards, dim=1, keepdim=True)
    std_rewards = torch.std(rewards, dim=1, keepdim=True)
    advantages = rewards - mean_rewards
    if normalize_by_std:
        advantages = advantages / (std_rewards + advantage_eps)
    advantages = advantages.view(-1)

    metadata = dict()
    metadata['std_rewards'] = torch.std(raw_rewards).item()
    metadata['mean_rewards'] = torch.mean(raw_rewards).item()
    metadata['max_rewards'] = torch.max(raw_rewards).item()
    metadata['min_rewards'] = torch.min(raw_rewards).item()
    return advantages, raw_rewards, metadata


def compute_naive_policy_gradient_loss(
    raw_rewards_or_advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
    ) -> torch.Tensor:
    """
    Compute the policy-gradient loss at every token, where raw_rewards_or_advantages is either
    the raw reward or an already-normalized advantage.
    Args:
    raw_rewards_or_advantages: torch.Tensor Shape (batch_size, 1), scalar
    reward/advantage for each rollout response.
    policy_log_probs: torch.Tensor Shape (batch_size, sequence_length), logprobs for
    each token.
    Returns:
    torch.Tensor Shape (batch_size, sequence_length), the per-token policy-gradient loss (to
    be aggregated across the batch and sequence dimensions in the training loop).
    """
    return raw_rewards_or_advantages * policy_log_probs

def compute_grpo_clip_loss(
    advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
    cliprange: float,
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """
    Args:
    advantages: torch.Tensor Shape (batch_size, 1), per-example advantages A.
    policy_log_probs: torch.Tensor Shape (batch_size, sequence_length), per-token log
    probs from the policy being trained.
    old_log_probs: torch.Tensor Shape (batch_size, sequence_length), per-token log probs
    from the old policy.
    cliprange: float Clip parameter ϵ (e.g. 0.2).
    Returns:
    tuple[torch.Tensor, dict[str, torch.Tensor]].
    loss torch.Tensor of shape (batch_size, sequence_length), the per-token clipped
    loss.
    metadata dict containing whatever you want to log. We suggest logging whether each
    token was clipped or not, i.e., whether the clipped"""
    ratio = torch.exp(policy_log_probs - old_log_probs)

    surrogant1 = advantages * ratio

    clipped = torch.clamp(ratio, 1 - cliprange, 1 + cliprange)
    surrogant2 = advantages * clipped
    per_token_loss = - torch.min(surrogant1, surrogant2)
    metadata = dict()
    clipped_mask = (clipped != ratio)
    metadata['clip_mask'] = clipped_mask
    

    return per_token_loss, metadata


def compute_policy_gradient_loss(
    policy_log_probs: torch.Tensor,
    loss_type: Literal["no_baseline", "reinforce_with_baseline", "grpo_clip"],
    raw_rewards: torch.Tensor | None = None,
    advantages: torch.Tensor | None = None,
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """
    Select and compute the desired policy-gradient loss.
    Args:
    policy_log_probs (batch_size, sequence_length), per-token log-probabilities from the
    policy being trained.
    loss_type One of "no_baseline", "reinforce_with_baseline", or "grpo_clip".
    raw_rewards Required if loss_type == "no_baseline"; shape (batch_size, 1).
    advantages Required for "reinforce_with_baseline" and "grpo_clip"; shape
    (batch_size, 1).
    old_log_probs Required for "grpo_clip"; shape (batch_size, sequence_length).
    cliprange Required for "grpo_clip"; scalar ϵ used for clipping.
    Returns:
    tuple[torch.Tensor, dict[str, torch.Tensor]].
    loss (batch_size, sequence_length), per-token loss.
    metadata dict, statistics from the underlying routine (e.g., clip fraction for GRPO-Clip)."""
    metadata_final = dict()
    if loss_type == 'no_baseline':
        assert raw_rewards is not None
        loss = compute_naive_policy_gradient_loss(raw_rewards,  policy_log_probs)
    if loss_type == 'reinforce_with_baseline':
        assert advantages is not None
        loss = compute_naive_policy_gradient_loss(advantages,  policy_log_probs)
    if loss_type == 'grpo_clip':
        assert advantages is not None
        assert old_log_probs is not None
        assert cliprange is not None
        loss, metadata = compute_grpo_clip_loss(
            advantages,
            policy_log_probs,
            old_log_probs, 
            cliprange,
        ) 
        metadata_final.update(metadata)
    return loss, metadata_final
    

def masked_mean(
    tensor: torch.Tensor,
    mask: torch.Tensor,
    dim: int | None = None,
    ) -> torch.Tensor:
    """
    Compute the mean of tensor along a given dimension, considering only those elements where
    mask == 1.
    Args:
    tensor: torch.Tensor The data to be averaged.
    mask: torch.Tensor Same shape as tensor; positions with 1 are included in the mean.
    dim: int | None Dimension over which to average. If None, compute the mean over all
    masked elements.
    Returns:
    torch.Tensor The masked mean; shape matches tensor.mean(dim) semantics."""
    return torch.sum(tensor * mask, dim=dim) / torch.sum(mask,dim=dim)
    

def grpo_microbatch_train_step(
    policy_log_probs: torch.Tensor,
    response_mask: torch.Tensor,
    gradient_accumulation_steps: int,
    loss_type: Literal["no_baseline", "reinforce_with_baseline", "grpo_clip"],
    raw_rewards: torch.Tensor | None = None,
    advantages: torch.Tensor | None = None,
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """
    Execute a forward-and-backward pass on a microbatch.
    Args:
    policy_log_probs (batch_size, sequence_length), per-token log-probabilities from the
    policy being trained.
    response_mask (batch_size, sequence_length), 1 for response tokens, 0 for
    prompt/padding.
    gradient_accumulation_steps Number of microbatches per optimizer step.
    loss_type One of "no_baseline", "reinforce_with_baseline", "grpo_clip".
    raw_rewards Needed when loss_type == "no_baseline"; shape (batch_size, 1).
    advantages Needed when loss_type != "no_baseline"; shape (batch_size, 1).
    old_log_probs Required for GRPO-Clip; shape (batch_size, sequence_length).
    cliprange Clip parameter ϵ for GRPO-Clip.
    Returns:
    tuple[torch.Tensor, dict[str, torch.Tensor]].
    loss scalar tensor. The microbatch loss, adjusted for gradient accumulation. We return
    this so we can log it.
    metadata Dict with metadata from the underlying loss call, and any other statistics you
    might want to log."""
    pertoken_loss, metadata = compute_policy_gradient_loss(
        policy_log_probs,
        loss_type,
        raw_rewards,
        advantages,
        old_log_probs,
        cliprange,
    )
    microbatch_loss = masked_mean(pertoken_loss, response_mask, dim=1).mean() / gradient_accumulation_steps
    microbatch_loss.backward()
    return microbatch_loss, metadata



def init_vllm(model_id: str, device: str, seed: int, gpu_memory_utilization: float = 0.85):
    """Start the inference process, here we use vLLM to hold a model on
    a GPU separate from the policy.
    """
    vllm_set_random_seed(seed)
    # Monkeypatch from TRL:
    # https://github.com/huggingface/trl/blob/
    # 22759c820867c8659d00082ba8cf004e963873c1/trl/trainer/grpo_trainer.py
    # Patch vLLM to make sure we can
    # (1) place the vLLM model on the desired device (world_size_patch) and
    # (2) avoid a test that is not designed for our setting (profiling_patch).
    world_size_patch = patch("torch.distributed.get_world_size", return_value=1)
    profiling_patch = patch(
    "vllm.worker.worker.Worker._assert_memory_footprint_increased_during_profiling",
    return_value=None
    )
    with world_size_patch, profiling_patch:
        return LLM(
        model=model_id,
        device=device,
        dtype=torch.bfloat16,
        enable_prefix_caching=True,
        gpu_memory_utilization=gpu_memory_utilization,
        
        )

def load_policy_into_vllm_instance(policy: PreTrainedModel, llm: LLM):
    """ Copied from https://github.com/huggingface/trl/blob/
    22759c820867c8659d00082ba8cf004e963873c1/trl/trainer/grpo_trainer.py#L670.
    """
    state_dict = policy.state_dict()
   # this lambda gets handed the exact HF model vLLM is using under the hood:
    llm.apply_model(lambda hf_model: hf_model.load_state_dict(policy_state, strict=False))


def grpo_train(
    policy_model, 
    llm_engine, 
    tokenizer, 
    reward_fn,
    dataloader, 
    rollout_batch_size, #rollout_batch_size = n_prompts_per_rollout_batch * group_size
    group_size, # #samples for each prompt
    train_batch_size, # can split to n chunks for rollout_batch, the size of each chunk
    epochs_per_rollout_batch, # for inner loop of whole rollout_batch data (how many turns to update current policy with same sampled response)
    gradient_accumulation_steps, 
    n_grpo_steps,  # outside loop , how many times to generate response basing on current policy
    cliprange,
    advantage_eps,
    normalize_by_std,
    sampling_params,
    optimizer, scheduler, device, dir_nm):
    start_time = time.time()

    policy_model = policy_model.to(device)
    policy_model.train()
    data_iter = iter(dataloader)

    assert rollout_batch_size % group_size == 0
    assert train_batch_size <= rollout_batch_size

    for outer in range(n_grpo_steps):
        # 1) Sample D_b via DataLoader ───────────────────
        try:
            prompts_batch, gts_batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            prompts_batch, gts_batch = next(data_iter)
        B_prompts = len(prompts_batch)  # = rollout_batch_size // group_size
        # 2) Snapshot old policy
        old_model = deepcopy(policy_model).to(device)
        old_model.eval()
        # 3) Sample G outputs per prompt with vLLM
        results = llm_engine.generate(
            prompts_batch,
            sampling_params=sampling_params  # must include num_return_sequences=group_size
        )
        # results is a list of length len(prompts_batch)
        all_responses = []
        for result in results:
            # result.outputs is a list of length=group_size
            for seq in result.outputs:
                all_responses.append(seq.text)
        repeated_gts = [gt for gt in gts_batch for _ in range(group_size)]

        # 4) compute group-normalized advantages
        advantages, raw_rewards, _ = compute_group_normalized_rewards(
            reward_fn,
            all_responses,
            repeated_gts,
            group_size,
            advantage_eps,
            normalize_by_std,
        )  # shape (rollout_batch_size,)
        # 5) tokenize prompt+response, build masks and labels
        prompt_strs = [p for p in prompts_batch for _ in range(group_size)]
        toks = tokenize_prompt_and_output(prompt_strs, all_responses, tokenizer)
        input_ids     = toks["input_ids"].to(device)
        labels        = toks["labels"].to(device)
        response_mask = toks["response_mask"].to(device)
        # 6) teacher-forcing scoring under old and new
        new_out = get_response_log_probs(policy_model, input_ids, labels)
        with torch.no_grad():
            old_out = get_response_log_probs(old_model, input_ids, labels)
        policy_log_probs = new_out["log_probs"]  # (B, L)
        old_log_probs    = old_out["log_probs"]  # (B, L)
        # 7) inner-loop: multiple updates per rollout batch
        # build dataset of fixed rollouts
        microbatch_size = train_batch_size // gradient_accumulation_steps
        ds = TensorDataset(
            policy_log_probs,
            old_log_probs,
            response_mask,
            advantages.unsqueeze(-1)
        )
        mb_loader = DataLoader(
            ds,
            batch_size=microbatch_size,
            shuffle=True,
            drop_last=True,
        )

        
        for epoch in range(epochs_per_rollout_batch):
            optimizer.zero_grad()
            for i, (mb_logp, mb_oldp, mb_mask, mb_adv) in enumerate(mb_loader):
                loss, stats = grpo_microbatch_train_step(
                    policy_log_probs=mb_logp,
                    response_mask=mb_mask,
                    gradient_accumulation_steps=gradient_accumulation_steps,
                    loss_type='grpo_clip',
                    raw_rewards=None,
                    advantages=mb_adv,
                    old_log_probs=mb_oldp,
                    cliprange=cliprange,
                )
                # Perform step at end of accumulation group
                if (idx + 1) % gradient_accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                    
    policy_model.save_pretrained(dir_nm)
    tokenizer.save_pretrained(dir_nm)



    # #for step in range(gradient_accumulation_steps):
    # model.train()
    # total_loss = 0
    # total_sample = 0
    # total_token_entropy = 0
    
    # for idx, batch in enumerate(tqdm(dataloader, total=len(dataloader))):
    #     batch_size = batch['input_ids'].size(0)
    #     input_ids = batch['input_ids'].to(device_)
    #     labels = batch['labels'].to(device_)
    #     response_masks = batch['response_mask'].to(device_)
    #     with torch.cuda.amp.autocast(dtype=torch.bfloat16):
    #         # prompt_texts = batch['prompt_text']
    #         # answer_texts = batch['answer_texts'] 
    #         report_cuda("Before forward")
    #         response = get_response_log_probs(model, input_ids, labels, True)
    #         report_cuda("after forward")

    #         log_probs, token_entropy = response['log_probs'], response['token_entropy']
    #         normalzied_token_entropy = token_entropy * response_masks #masked_normalize(tensor=token_entropy, mask=response_masks, dim=1, normalize_constant=1.0) 
    #         curr_batch_response_len = torch.sum(response_masks,dim=1) #(batch,)
    #         curr_batch_response_entropy = torch.sum(normalzied_token_entropy, dim=1) #(batch,)
    #         curr_batch_response_avg_entropy = curr_batch_response_entropy / curr_batch_response_len  #(batch,)
    #         report_cuda("after entroy calculation")
    #         token_entropy = token_entropy.detach().cpu()
    #         normalzied_token_entropy = normalzied_token_entropy.detach().cpu()
    #         curr_batch_response_len = curr_batch_response_len.detach().cpu()  
    #         curr_batch_response_entropy = curr_batch_response_entropy.detach().cpu()
    #         curr_batch_response_avg_entropy = curr_batch_response_avg_entropy.detach().cpu() 
    #         report_cuda("before loss backward")
    #         loss, metadata = sft_microbatch_train_step(
    #             log_probs, 
    #             response_masks,
    #             gradient_accumulation_steps,
    #             1.0)
    #         report_cuda("after loss backward")
    #     total_loss+=loss.detach().item()
    #     total_sample += batch_size
           
    #     total_token_entropy += torch.sum(curr_batch_response_avg_entropy).item()
        

    #     if (idx + 1) % gradient_accumulation_steps == 0:
    #         # Clip the norm of the gradients to 1.0.
    #         # This is to help prevent the "exploding gradients" problem.
    #         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    #         # Update weights every `gradient_accumulation_steps` batches.
    #         optimizer_.step()
    #         scheduler_.step()
    #         # Zero gradients every `gradient_accumulation_steps` batches.
    #         optimizer_.zero_grad()
    #         torch.cuda.empty_cache()
    # end_time = time.time()
    # time_elapes = (end_time - start_time) / 60
    # print(f"avg_epoch_loss: {total_loss / total_sample:.2f}, avg_epoch_token_entropy: {total_token_entropy / total_sample :.2f}, time spent: {time_elapes : .2f} min")
    # torch.save(model.state_dict(), dir_nm)

    # #torch.cuda.empty_cache()
    # del model
    # torch.cuda.empty_cache()
    # gc.collect()
    # report_cuda("final after training current epoch")
    


SyntaxError: incomplete input (1069838208.py, line 137)

In [4]:

ground_truths = [[gt]*2 for gt in ['1','b']]
ground_truths = list(itertools.chain.from_iterable(ground_truths))
ground_truths

['1', '1', 'b', 'b']